# Week 11 Student Notebook: Multi-Specialist + MCP

**Your Goal:** Build a multi-specialist workflow using MCP servers - same architecture as Master Notebook!

**How to Use This Notebook:**
1. Follow each TODO in order (Steps 1-7)
2. Reference the Master Notebook for how things work
3. Fill in the code cells with implementations
4. Test as you go - don't skip the demos!

**Learning Outcomes:**
- ✅ Connect agents to external MCP servers
- ✅ Build 6 specialist agents (supervisor, math, research, analysis, writer, final)
- ✅ Create conditional routing logic with supervisor decisions
- ✅ Chain specialists together for complex workflows
- ✅ Understand async/await for network I/O

## Prerequisites: Start MCP Servers

**IMPORTANT:** You must run 4 MCP servers BEFORE executing this notebook. Open 4 separate terminals:

```powershell
# Terminal 1
python 1_add_numbers_server.py

# Terminal 2
python 2_multiply_numbers_server.py

# Terminal 3
python 3_get_current_time_server.py

# Terminal 4
python 4_rag_server.py
```

**Verify:** Each terminal should show "Listening on http://localhost:XXXX"

**Why separate terminals?** Because in production, these are separate microservices running independently. Separate terminals simulate that architecture!

## Step 1: Imports and Environment

**What you need to import:**
- Environment loader (dotenv)
- Async/concurrency library
- JSON for parsing supervisor decisions
- LangGraph for state machine
- LangChain for messages and LLM
- MCP client libraries for connecting to servers

**Hint:** Check the Master Notebook cell "Setup and Imports" to see which libraries are needed and why each one matters.

**Teaching Point:** Notice we need `asyncio` because MCP servers use async/await for efficient network communication!

In [ ]:
# STEP 1: Imports and Environment Setup
# HINTS:
#   - dotenv: load_dotenv() loads .env file (API keys, config)
#   - asyncio: enables async/await for concurrent operations
#   - json: parse supervisor's JSON decision
#   - Literal: type hint for fixed set of values
#   - StateGraph, MessagesState: LangGraph components
#   - HumanMessage, AIMessage, ToolMessage: message types for agent history
#   - ChatOpenAI: LLM for supervisor agent
#   - ClientSession, sse_client: MCP client for calling remote servers
#
# REFER TO: Master Notebook "Step 1" and "Step 2" headers
#
# TODO: Fill in the imports based on hints above

## Step 2: Initialize the LLM

**What you need:**
- Only the SUPERVISOR agent uses the LLM (specialty: classification)
- All other agents are deterministic (no LLM needed)
- Use gpt-4o-mini for fast, efficient classification
- Set temperature=0 for consistent, deterministic decisions

**Why only supervisor uses LLM?**
- LLM calls are expensive and slow
- Supervisor makes ONE decision per question
- Specialists execute that decision (no thinking needed)

**Hint:** Look at Master Notebook "Initialize the LLM" section

In [ ]:
# STEP 2: Initialize LLM for Supervisor Agent
# HINTS:
#   - Use ChatOpenAI from langchain_openai
#   - model: "gpt-4o-mini" (fast, cheap, good for classification)
#   - temperature: 0 (deterministic, no randomness)
#   - Only SUPERVISOR uses this LLM (teaching point!)
#
# REFER TO: Master Notebook "Initialize the LLM" cell
#
# TODO: Create llm variable with ChatOpenAI

## Step 3: MCP Configuration & Helper Functions

**What you need to build:**

### 3a. MCP_SERVERS Dictionary
Map tool names to HTTP endpoints:
- "add" → http://localhost:8000/sse
- "multiply" → http://localhost:8001/sse
- "time" → http://localhost:8002/sse
- "rag" → http://localhost:8003/sse

**Why HTTP?** MCP servers are separate processes. We call them over the network using SSE protocol.

### 3b. extract_text_from_result(result)
MCP servers return complex nested objects. This function extracts the readable text.
- Input: MCP result object with `.content` attribute
- Output: String (the actual answer)

### 3c. async call_mcp_tool(server_url, tool_name, arguments)
This is the CORE MCP client function:
1. Connect to MCP server via SSE
2. Initialize ClientSession
3. Call the tool with arguments
4. Return result

**Hint:** Check Master Notebook "MCP Server Configuration" section for full implementations.

In [ ]:
# STEP 3: MCP Configuration & Helper Functions
#
# 3a. MCP_SERVERS dictionary
#     Maps tool names to their HTTP endpoints
#     HINTS:
#       - "add": port 8000
#       - "multiply": port 8001
#       - "time": port 8002
#       - "rag": port 8003
#       - All use /sse path
#
# 3b. extract_text_from_result(result: Any) -> str
#     MCP responses have nested structure: result.content = [{"type": "text", "text": "answer"}]
#     Extract just the text part
#     HINTS:
#       - Check if result.content is a list
#       - Loop through items to find text
#       - Join multiple parts with newlines
#       - Fallback to str(content) if it's a simple type
#
# 3c. async call_mcp_tool(server_url, tool_name, arguments)
#     Call a remote MCP tool over HTTP
#     HINTS:
#       - Use sse_client as a context manager
#       - Initialize ClientSession inside
#       - Call session.call_tool(tool_name, arguments)
#       - Return the result
#
# REFER TO: Master Notebook "MCP Server Configuration" section
#
# TODO: Implement all three components above

## Step 4: Specialist Agents (6 Total!)

**Architecture Overview:**
```
Supervisor (classifier, uses LLM)
├── Math Specialist (calls MCP servers 8000-8001)
└── Research Specialist (calls MCP server 8003)
    ├── Analysis Specialist (calls MCP server 8002)
    └── Writer Specialist (no MCP, just formatting)
└── Final Response (chooses output format)
```

**What each agent does:**

| Agent | Input | Output | MCP Call? | LLM? |
|-------|-------|--------|-----------|------|
| **Supervisor** | User question | JSON routing decision | ❌ | ✅ |
| **Math Specialist** | supervisor.decision | math_result | ✅ (8000-8001) | ❌ |
| **Research Specialist** | messages | research_result | ✅ (8003) | ❌ |
| **Analysis Specialist** | research_result | analysis_summary | ✅ (8002) | ❌ |
| **Writer Specialist** | analysis_summary | report | ❌ | ❌ |
| **Final Response** | math_result OR report | final_answer | ❌ | ❌ |

**Key Teaching Points:**
- All agents are async functions
- All take `state: dict` and return modified `state`
- Agents communicate through shared state (the "clipboard")
- Only supervisor uses LLM

**Hints:** 
- See Master Notebook sections for each agent
- Each has detailed docstrings explaining what it does
- Use `state.get()` to read previous agent's output
- Use `state[key] = value` to write your output

In [ ]:
# STEP 4: Implement 6 Specialist Agents
#
# AGENT 1: supervisor_agent(state) 
#   HINTS:
#     - Read: messages from state
#     - Do: Ask LLM what type of question it is
#     - Return: JSON with decision (route_to_math / route_to_research / cannot_help)
#   REFER TO: Master Notebook "Supervisor Agent" section
#
# AGENT 2: research_specialist_agent(state)
#   HINTS:
#     - Read: messages[0].content (user's original question)
#     - Do: await call_mcp_tool(MCP_SERVERS["rag"], "ask_question", {query: ...})
#     - Write: research_result to state
#   REFER TO: Master Notebook "Research Specialist Agent" section
#
# AGENT 3: analysis_specialist_agent(state)
#   HINTS:
#     - Read: research_result from state
#     - Do: await call_mcp_tool(MCP_SERVERS["time"], "get_current_time", {})
#     - Write: Combine research_result + timestamp → analysis_summary
#   REFER TO: Master Notebook "Analysis Specialist Agent" section
#
# AGENT 4: writer_specialist_agent(state)
#   HINTS:
#     - Read: analysis_summary from state
#     - Do: Format with borders, headers, sections
#     - Write: report to state
#     - NOTE: No MCP call! Just formatting!
#   REFER TO: Master Notebook "Writer Specialist Agent" section
#
# AGENT 5: math_specialist_agent(state)
#   HINTS:
#     - Read: supervisor_decision from state (operation, parameters)
#     - Do: await call_mcp_tool(..., operation, parameters)
#     - Write: math_result to state
#   REFER TO: Master Notebook "Math Specialist Agent" section
#
# AGENT 6: final_response_agent(state)
#   HINTS:
#     - Read: math_result OR report (depending on what ran)
#     - Do: Choose which to return
#     - Write: final_answer to state
#   REFER TO: Master Notebook "Final Response Agent" section
#
# TODO: Implement all 6 async agent functions

## Step 5: State Definition & Routing Logic

**5a. OrchestratorState Class**

This is the "clipboard" all agents share:

| Field | Type | Used By |
|-------|------|---------|
| `messages` | List[BaseMessage] | All agents (inherited from MessagesState) |
| `supervisor_decision` | dict | Supervisor writes, Math Specialist reads |
| `math_result` | Any | Math Specialist writes, Final Response reads |
| `research_result` | str | Research Specialist writes, Analysis reads |
| `analysis_summary` | str | Analysis writes, Writer reads |
| `report` | str | Writer writes, Final Response reads |
| `final_answer` | str | Final Response writes (final output) |

**5b. route_after_supervisor Function**

This is the "decision router":
- Input: `state` (contains supervisor_decision)
- Output: Node name (string: "math_specialist" OR "research_specialist" OR "final_response")
- Logic: Read decision from state, return appropriate next agent name

**Why separate routing?**
- LangGraph needs a function to determine next step
- Makes routing logic explicit and testable
- Easy to extend (add more routes as needed)

**Hints:**
- Use `Literal["math_specialist", "research_specialist", "final_response"]` as return type
- Check `state["supervisor_decision"]["decision"]` value
- Return the corresponding agent name

**REFER TO:** Master Notebook "State" and "Routing" sections

In [ ]:
# STEP 5: State & Routing
#
# 5a. OrchestratorState class
#   Base: MessagesState (from LangGraph)
#   Add fields: supervisor_decision, math_result, research_result, analysis_summary, report, final_answer
#   HINTS:
#     - Inherit from MessagesState
#     - Use Type hints: dict, Any, str, etc.
#     - Provide defaults: {} or "" or None
#   REFER TO: Master Notebook "Orchestrator State" section
#
# 5b. route_after_supervisor function
#   Input: state (OrchestratorState)
#   Output: Literal["math_specialist", "research_specialist", "final_response"]
#   HINTS:
#     - Get supervisor_decision from state
#     - Check decision["decision"] value:
#       - "route_to_math" → return "math_specialist"
#       - "route_to_research" → return "research_specialist"
#       - "cannot_help" → return "final_response"
#   REFER TO: Master Notebook "Routing Function" section
#
# TODO: Implement OrchestratorState class and route_after_supervisor function

## Step 6: Build the Orchestrator (LangGraph)

**What you're building:** A state machine that coordinates all 6 agents.

**High-level structure:**
```
graph = StateGraph(OrchestratorState)

// Add all agent nodes
graph.add_node("supervisor", supervisor_agent)
... (add other 5 agents) ...

// Set where to start
graph.set_entry_point("supervisor")

// Add conditional routing (supervisor decides where to go)
graph.add_conditional_edges("supervisor", route_after_supervisor, {...})

// Connect agents in sequence
graph.add_edge("math_specialist", "final_response")
graph.add_edge("research_specialist", "analysis_specialist")
... (connect to research chain) ...

// Set where to end
graph.set_finish_point("final_response")

// Compile and return
return graph.compile()
```

**Key Concepts:**
- **Nodes:** Agent functions
- **Edges:** "Go to this next agent after this one finishes"
- **Conditional Edges:** "Choose next agent based on decision"
- **Entry/Finish:** Where graph starts and ends

**For math questions:**
supervisor → math_specialist → final_response

**For research questions:**
supervisor → research_specialist → analysis_specialist → writer_specialist → final_response

**For out-of-scope questions:**
supervisor → final_response

**REFER TO:** Master Notebook "Build the Orchestrator" section

In [ ]:
# STEP 6: Build Orchestrator (LangGraph State Machine)
#
# PSEUDO-CODE (fill in the details):
# def build_orchestrator():
#   1. Create StateGraph with OrchestratorState
#   2. add_node for each agent:
#      - "supervisor" → supervisor_agent
#      - "math_specialist" → math_specialist_agent
#      - "research_specialist" → research_specialist_agent
#      - "analysis_specialist" → analysis_specialist_agent
#      - "writer_specialist" → writer_specialist_agent
#      - "final_response" → final_response_agent
#   3. set_entry_point("supervisor")
#   4. add_conditional_edges from supervisor:
#      - Maps "math_specialist" to math_specialist node
#      - Maps "research_specialist" to research_specialist node
#      - Maps "final_response" to final_response node
#   5. Connect specialist flows:
#      - math_specialist → final_response
#      - research_specialist → analysis_specialist
#      - analysis_specialist → writer_specialist
#      - writer_specialist → final_response
#   6. set_finish_point("final_response")
#   7. return graph.compile()
#
# HINTS:
#   - graph.compile() returns the executable workflow
#   - add_conditional_edges needs: (source_node, routing_function, mapping_dict)
#   - mapping_dict maps routing function return values to node names
#
# REFER TO: Master Notebook "Build the Orchestrator" section
#
# TODO: Implement def build_orchestrator() function

## Step 7: Run Tests & See Everything Work!

**Test Cases to Run:**

| Question | Type | Path | Expected Flow |
|----------|------|------|------|
| "What is 5 plus 3?" | Math | supervisor → math → final | Result: 8 |
| "Multiply 7 and 6" | Math | supervisor → math → final | Result: 42 |
| "What is machine learning?" | Research | supervisor → research → analysis → writer → final | Result: Report |
| "What's your favorite color?" | Out-of-scope | supervisor → final | Result: "Can't solve" |

**What you'll see in terminal output:**
```
============================================================================
📢 Question: What is 5 plus 3?
============================================================================
🎯 SUPERVISOR: Analyzing question...
   Decision: route_to_math
🧮 MATH SPECIALIST: Calling MCP math server...
   MCP Tool: add_numbers
   Parameters: {'a': 5, 'b': 3}
   MCP Result: 8
📝 FINAL RESPONSE: Formatting answer...
   Answer ready

============================================================================
✅ Final Answer: The answer is: 8
============================================================================
```

**How to test:**
```python
async def run_orchestrator(user_question: str):
    app = build_orchestrator()
    initial_state = OrchestratorState(messages=[HumanMessage(content=user_question)])
    final_state = await app.ainvoke(initial_state)
    return final_state
```

**Run it:**
```python
for question in test_questions:
    await run_orchestrator(question)
```

**REFER TO:** Master Notebook "Run Tests" section

In [ ]:
# STEP 7: Run Tests & Demos
#
# 7a. async run_orchestrator(user_question: str) function
#   HINTS:
#     - Build orchestrator: app = build_orchestrator()
#     - Create initial state with user question: OrchestratorState(messages=[HumanMessage(content=...)])
#     - Run async: final_state = await app.ainvoke(initial_state)
#     - Print final_state["final_answer"]
#   REFER TO: Master Notebook "Run Tests" section
#
# 7b. Test questions to run:
#   - "What is 5 plus 3?" (math test)
#   - "Can you multiply 7 and 6?" (math test)
#   - "What is machine learning?" (research test)
#   - "What's your favorite color?" (out-of-scope test)
#
# HINTS:
#   - Use await run_orchestrator(question) for each
#   - Watch the output to see agents running in sequence
#   - Check MCP server terminals to see requests coming in!
#
# TODO: Implement run_orchestrator() and run all tests